# ◈ ScenePilot AI — Master Execution Notebook

> **Full pipeline walkthrough: system setup → agents → diff-patch repair → budget gates → sandbox → API → Docker deployment → end-to-end test suite**

Built with LangGraph · FastAPI · React · Groq llama-3.3-70b · Gemini 2.5 Flash · IBM Bob

---

## Execution Rules

| Rule | Detail |
|---|---|
| **Run cells top-to-bottom** | Every cell depends on the state produced by cells above it |
| **`%%writefile` cells** | Write the exact production source file to disk |
| **Shell cells (`!`)** | Run OS commands in the notebook kernel's working directory |
| **Python cells** | Execute in-process logic, unit tests, validation routines |
| **Prerequisites** | Python 3.11+, Docker Desktop running, valid `.env` with API keys |

---

### Section Map

```
SECTION 1  ── System Setup & Directory Foundations          (Cells  1 –  4)
SECTION 2  ── Architecture & Core Data Models               (Cells  5 –  7)
SECTION 3  ── Multi-Agent Pipeline (incl. Repair Mode)      (Cells  8 – 13)
SECTION 4  ── Sandbox Validator + invalid_edges             (Cells 14 – 15)
SECTION 5  ── FastAPI Gateway + Pre-flight Estimator        (Cells 16 – 17)
SECTION 6  ── Docker Infrastructure & Deployment            (Cells 18 – 20)
SECTION 7  ── End-to-End Test Suite (A · B · C · D)         (Cells 21 – 24)
```

---
## 📁 SECTION 1 — System Setup & Directory Foundations

In [ ]:
# Cell 1 — Create full project directory tree (idempotent)
!mkdir -p agents api core data/rules data/samples frontend/src/components frontend/src/hooks frontend/src/types sandbox prometheus

In [ ]:
%%writefile requirements.txt
# ── Web framework ─────────────────────────────────────────────────────────────
fastapi==0.111.0
uvicorn[standard]==0.29.0
python-dotenv==1.0.1
pydantic==2.7.1

# ── LangGraph / LangChain ─────────────────────────────────────────────────────
langgraph==0.1.5
langchain-core==0.2.5

# ── LLM clients ───────────────────────────────────────────────────────────────
groq==0.9.0
google-generativeai==0.7.2

# ── Embeddings + FAISS ────────────────────────────────────────────────────────
sentence-transformers==3.0.1
faiss-cpu==1.8.0

# ── Graph / validation ────────────────────────────────────────────────────────
networkx==3.3

# ── JSON repair ───────────────────────────────────────────────────────────────
json-repair==0.25.2

# ── Observability ─────────────────────────────────────────────────────────────
prometheus-client==0.20.0
opentelemetry-api==1.24.0
opentelemetry-sdk==1.24.0
opentelemetry-exporter-otlp-proto-grpc==1.24.0

# ── Utilities ─────────────────────────────────────────────────────────────────
httpx==0.27.0

In [ ]:
%%writefile .env.example
# ── LLM API Keys ──────────────────────────────────────────────────────────────
GROQ_API_KEY=your_groq_api_key_here
GEMINI_API_KEY=your_gemini_api_key_here

# ── Story generation settings ─────────────────────────────────────────────────
# TOKEN_BUDGET_LIMIT: total token ceiling per pipeline run.
#   Small premise   (< 100 chars)  ->  5,000  is usually enough
#   Standard premise (100-300 chars) ->  10,000
#   Complex premise  (300-700 chars) ->  20,000   (recommended for rich stories)
TOKEN_BUDGET_LIMIT=20000
MAX_RETRIES=3
# SCENE_MULTIPLIER: controls the pre-flight output-token estimate.
# Default 40 is calibrated to complex thriller/sci-fi premises (~18-22 scenes).
# Lower to 15-20 for educational/marketing genres with shorter outputs.
SCENE_MULTIPLIER=40

# ── Style Vault ───────────────────────────────────────────────────────────────
STYLE_SIMILARITY_THRESHOLD=0.35

# ── Sandbox ───────────────────────────────────────────────────────────────────
SANDBOX_USE_DOCKER=false
SANDBOX_DOCKER_IMAGE=python:3.11-slim

# ── CORS ──────────────────────────────────────────────────────────────────────
CORS_ORIGINS=http://localhost:5173,http://localhost:3000

# ── OpenTelemetry (optional) ──────────────────────────────────────────────────
# OTEL_EXPORTER_OTLP_ENDPOINT=http://localhost:4317

In [ ]:
# Cell 4 — Install dependencies + verify critical imports
#
# Windows (PowerShell):
#   python -m venv .venv
#   .venv\Scripts\activate.ps1
#   copy .env.example .env   <- then fill in your API keys
#
# macOS / Linux:
#   python3 -m venv .venv && source .venv/bin/activate
#   cp .env.example .env

!pip install -q -r requirements.txt
print("Dependencies installed.")

import importlib
for pkg in ["fastapi", "langgraph", "networkx", "faiss", "groq", "prometheus_client", "json_repair"]:
    try:
        importlib.import_module(pkg)
        print(f"  OK  {pkg}")
    except ImportError as e:
        print(f"  MISSING  {pkg}: {e}")

---
## 🗺️ SECTION 2 — Architecture & Core Data Models

### Cell 5 — Architecture & State Machine Routing

```
BROWSER  http://localhost:5173
React 18 + Vite | React Flow | Recharts | Blueprint SVG | Emulator
         |
         |  HTTP  (nginx reverse proxy, 120s timeout)
         v
FASTAPI  :8000
  POST /api/generate  ->  run_pipeline()         [pre-flight budget estimator]
  POST /api/validate  ->  validate_story()        [no LLM]
  POST /api/blueprint ->  generate_blueprint()
  GET  /api/samples   ->  demo library
  GET  /metrics       ->  Prometheus
         |
         v
LANGGRAPH  StateGraph(ScenePilotState)

  [generate] --> [style_vault] --> [sandbox]
                                       |
                    approved ----------+
                       |    retry_count < max
                       |    AND _has_budget_for_retry() -----> [retry] --> [generate]
                       |                                           (repair_mode if
                       |                                            broken_nodes present)
                       |    else fail --------------------------> [fail]
                       v                                              |
                 [compliance] <----------------------------------------
                       |
                      END
```

#### Token Budget Gate — three layers

| Layer | Location | Action |
|---|---|---|
| Pre-flight | `api/routes.py` | Estimates worst-case before any LLM call; rejects with exact `TOKEN_BUDGET_LIMIT=N` recommendation |
| Router gate | `agents/orchestrator.py` | `_has_budget_for_retry()` blocks retry if remaining < reserve (1,200 repair / 9,500 full-gen) |
| Generator guard | `agents/story_generator.py` | Defence-in-depth: refuses LLM call; surfaces `last_story` as best-effort result |

#### Routing decision table

| `approved` | `retry_count < max` | `_has_budget_for_retry()` | Route |
|:---:|:---:|:---:|---|
| True | — | — | `compliance → END` |
| False | True | True | `retry → generate` (repair or full-gen) |
| False | True | False | `fail → compliance → END` (budget halt) |
| False | False | — | `fail → compliance → END` (retries exhausted) |

In [ ]:
%%writefile agents/state.py
"""
ScenePilotState — shared TypedDict passed through every LangGraph node.
"""
from __future__ import annotations

from typing import Any, Optional
from typing_extensions import TypedDict


class ValidationResult(TypedDict):
    passed: bool
    issues: list[str]
    cycles_detected: int
    schema_errors: list[str]
    style_violations: list[str]


class AuditEntry(TypedDict):
    story_id: str
    fingerprint: str
    timestamp: str
    agent_spans: list[dict[str, Any]]
    token_spend: int
    validation: ValidationResult


class ScenePilotState(TypedDict):
    # Input
    story_id: str
    premise: str
    genre: str          # thriller | fantasy | sci-fi | educational | marketing
    tone: float         # 0.0 (dark) -> 1.0 (light)

    # Story payload
    story: Optional[dict[str, Any]]          # raw JSON from StoryGeneratorAgent
    story_json: Optional[str]                # serialised string for sandbox

    # Validation state
    validation: Optional[ValidationResult]
    style_check: Optional[dict[str, Any]]    # raw FAISS result

    # Control flow
    approved: bool
    retry_count: int
    max_retries: int

    # Diff-based repair state
    # broken_nodes: exact (source, target) cycle edge pairs from sandbox
    broken_nodes: Optional[list[tuple[str, str]]]
    # last_story: full story from most recent generation — patched on repair
    last_story: Optional[dict[str, Any]]
    # repair_mode: True when generator is running a diff-patch pass
    repair_mode: bool

    # Audit / telemetry
    audit: Optional[AuditEntry]
    agent_spans: list[dict[str, Any]]
    token_spend: int
    error: Optional[str]
    # budget_halt: True when mid-pipeline or generator budget guard fired
    budget_halt: bool

In [ ]:
%%writefile core/telemetry.py
"""
Prometheus metrics + OpenTelemetry tracer setup.
"""
from __future__ import annotations

from prometheus_client import Counter, Histogram  # type: ignore

STORIES_GENERATED = Counter(
    "scenepilot_stories_generated_total",
    "Total approved stories generated",
)
VALIDATION_DURATION = Histogram(
    "scenepilot_validation_duration_seconds",
    "Time spent in sandbox validation",
    buckets=[0.05, 0.1, 0.25, 0.5, 1.0, 2.5, 5.0],
)
LOOP_DETECTIONS = Counter(
    "scenepilot_loop_detections_total",
    "Number of cycle/loop detections in story graphs",
)
STYLE_VIOLATIONS = Counter(
    "scenepilot_style_violations_total",
    "FAISS tone/format violations detected",
)
SANDBOX_REJECTIONS = Counter(
    "scenepilot_sandbox_rejections_total",
    "Stories rejected by the sandbox validator",
)
AGENT_TOKEN_SPEND = Counter(
    "scenepilot_agent_token_spend_total",
    "Total LLM tokens spent across all agents",
)
BUDGET_HALTS = Counter(
    "scenepilot_budget_halts_total",
    "Times the token budget ceiling was hit",
)


def get_tracer(name: str = "scenepilot"):
    try:
        from opentelemetry import trace  # type: ignore
        return trace.get_tracer(name)
    except ImportError:
        return None


def setup_otel() -> None:
    try:
        import os
        from opentelemetry import trace  # type: ignore
        from opentelemetry.sdk.trace import TracerProvider  # type: ignore
        from opentelemetry.sdk.trace.export.otlp.proto.grpc.trace_exporter import OTLPSpanExporter  # type: ignore
        from opentelemetry.sdk.trace.export import BatchSpanProcessor  # type: ignore
        endpoint = os.environ.get("OTEL_EXPORTER_OTLP_ENDPOINT")
        if not endpoint:
            return
        provider = TracerProvider()
        provider.add_span_processor(BatchSpanProcessor(OTLPSpanExporter(endpoint=endpoint)))
        trace.set_tracer_provider(provider)
    except Exception:
        pass

---
## 🤖 SECTION 3 — Multi-Agent Pipeline (incl. Repair Mode & Budget Gates)

In [ ]:
%%writefile core/utils.py
"""
core/utils.py — merge_patch helper for diff-based repair.

Takes the original full story (last_story from state) and a minimal patch
object returned by the LLM in REPAIR MODE, and returns a new story dict
with only the named scenes' choices arrays replaced.
"""
from __future__ import annotations

import copy
from typing import Any


def merge_patch(
    original_story: dict[str, Any],
    patch: dict[str, Any],
) -> dict[str, Any]:
    """Deeply merge a repair patch into original_story.

    patch schema: { "scene_id": [new_choices_list], ... }

    Only the 'choices' array of each named scene is replaced.
    All scene text, tone, id, and unlisted scenes are preserved verbatim.
    The original dict is never mutated — a deep copy is returned.
    """
    if not patch:
        return original_story

    merged = copy.deepcopy(original_story)
    scenes = merged.get("scenes")
    if not isinstance(scenes, list):
        return merged

    # O(1) lookup index
    scene_index: dict[str, dict[str, Any]] = {
        s["id"]: s for s in scenes if isinstance(s, dict) and "id" in s
    }

    for scene_id, new_choices in patch.items():
        scene = scene_index.get(scene_id)
        if scene is None or not isinstance(new_choices, list):
            continue
        scene["choices"] = new_choices

    return merged

In [ ]:
%%writefile agents/story_generator.py
"""
StoryGeneratorAgent — Groq (primary) or Gemini (fallback).

Two execution modes:
  FULL GENERATION  (retry_count == 0 OR no broken_nodes)
    -> Full system prompt, max_tokens=8192, produces complete story JSON.

  REPAIR MODE  (retry_count >= 1 AND broken_nodes present AND last_story exists)
    -> Sends only the broken scenes + validator issues, max_tokens=1024.
    -> LLM returns a minimal patch: {scene_id: [new_choices]}.
    -> core.utils.merge_patch() applies patch to last_story.
    -> Token cost: ~600 vs ~9,000 for full generation (>80% saving).

Defence-in-depth budget guard fires before any LLM call:
  if remaining < reserve -> abort, surface last_story as best-effort result.
"""
from __future__ import annotations

import json
import os
import time
from typing import Any

from agents.state import ScenePilotState
from core.utils import merge_patch

_REPAIR_RESERVE: int = 1_200
_FULL_GEN_RESERVE: int = 9_500


def _groq_client():
    from groq import Groq  # type: ignore
    return Groq(api_key=os.environ["GROQ_API_KEY"])


def _gemini_client():
    import google.generativeai as genai  # type: ignore
    genai.configure(api_key=os.environ["GEMINI_API_KEY"])
    return genai.GenerativeModel("gemini-2.5-flash")


SYSTEM_PROMPT = """You are ScenePilot, an expert interactive narrative designer.
Given a story premise, genre, and tone score (0=dark, 1=light), output ONLY a
valid JSON object matching this exact schema -- no markdown fences, no commentary:

{
  "title": "<story title>",
  "genre": "<genre>",
  "scenes": [
    {
      "id": "scene_001",
      "text": "<scene description>",
      "tone": "<tense|hopeful|dark|neutral|playful>",
      "choices": [
        {"text": "<choice label>", "next": "<scene_id or null for ending>"}
      ]
    }
  ]
}

Rules:
- Generate 12-20 scenes for a rich branching narrative.
- Every non-ending scene must have 2-3 choices.
- Ending scenes have an empty choices array [].
- scene ids are scene_001 ... scene_NNN (zero-padded to 3 digits).
- CRITICAL: DAG only. A choice 'next' must ALWAYS point to a HIGHER scene number.
- Tone: <=0.3 -> dark, 0.3-0.7 -> tense/neutral, >0.7 -> hopeful/playful.
- COMPACTNESS: scene text 1-2 sentences max. Choice labels 3-6 words.
"""

REPAIR_SYSTEM_PROMPT = """You are ScenePilot in REPAIR MODE.
You will receive an existing branching narrative and a list of cycle violations.

Output ONLY a JSON patch object -- no markdown fences, no commentary:
{ "<scene_id>": [{"text": "<label>", "next": "<scene_id or null>"}, ...], ... }

Rules:
- Do NOT rewrite scene text, tone, or id.
- Do NOT alter any scene NOT in the broken_nodes list.
- Each patched scene's choices MUST point to a HIGHER scene number (forward only).
- Return ONLY the patch object.
"""


def _build_user_prompt(premise: str, genre: str, tone: float) -> str:
    return (
        f"Premise: {premise}\n"
        f"Genre: {genre}\n"
        f"Tone score: {tone:.2f}\n\n"
        "Generate the full branching narrative JSON now."
    )


def _build_repair_prompt(
    story: dict[str, Any],
    broken_nodes: list[tuple[str, str]],
    validation_issues: list[str],
) -> str:
    broken_ids: set[str] = set()
    for src, dst in broken_nodes:
        broken_ids.add(src)
        broken_ids.add(dst)
    relevant_scenes = [s for s in story.get("scenes", []) if s.get("id") in broken_ids]
    return (
        "EXISTING STORY (broken scenes only):\n"
        + json.dumps({"scenes": relevant_scenes}, indent=2)
        + "\n\nCYCLE VIOLATIONS:\n"
        + "\n".join(f"  - {i}" for i in validation_issues)
        + "\n\nBROKEN EDGES:\n"
        + json.dumps(broken_nodes)
        + "\n\nReturn the patch JSON now."
    )


def _call_groq(system: str, user: str, max_tokens: int = 8192) -> tuple[str, int]:
    client = _groq_client()
    response = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[
            {"role": "system", "content": system},
            {"role": "user",   "content": user},
        ],
        temperature=0.7,
        max_tokens=max_tokens,
    )
    content = response.choices[0].message.content or ""
    tokens  = response.usage.total_tokens if response.usage else 0
    return content, tokens


def _call_gemini(system: str, user: str, max_tokens: int = 8192) -> tuple[str, int]:
    client   = _gemini_client()
    response = client.generate_content(
        system + "\n\n" + user,
        generation_config={"temperature": 0.7, "max_output_tokens": max_tokens},
    )
    content = response.text or ""
    tokens  = getattr(getattr(response, "usage_metadata", None), "total_token_count", 0) or 0
    return content, tokens


def _strip_fences(text: str) -> str:
    text = text.strip()
    if text.startswith("```"):
        lines = text.split("\n")
        text  = "\n".join(lines[1:-1]) if lines[-1].strip() == "```" else "\n".join(lines[1:])
    return text


def _parse_json(raw: str) -> dict[str, Any]:
    text = _strip_fences(raw)
    try:
        return __import__("json").loads(text)
    except __import__("json").JSONDecodeError:
        try:
            from json_repair import repair_json  # type: ignore
            return __import__("json").loads(repair_json(text))
        except Exception:
            raise


def _break_cycles(story: dict[str, Any]) -> dict[str, Any]:
    """Remove back-edges using scene position ordering."""
    scenes = story.get("scenes")
    if not scenes or not isinstance(scenes, list):
        return story
    order = {s.get("id"): i for i, s in enumerate(scenes) if s.get("id")}
    for scene in scenes:
        choices = scene.get("choices")
        if not isinstance(choices, list):
            continue
        src = order.get(scene.get("id"), -1)
        scene["choices"] = [
            c for c in choices
            if c.get("next") is None or order.get(c.get("next"), src + 1) > src
        ]
    return story


def story_generator_node(state: ScenePilotState) -> ScenePilotState:
    span_start  = time.time()
    error: str | None = None
    story: dict[str, Any] | None = None
    tokens = 0

    retry_count  = state.get("retry_count", 0)
    broken_nodes = state.get("broken_nodes") or []
    last_story   = state.get("last_story")
    repair_mode  = retry_count >= 1 and bool(broken_nodes) and last_story is not None

    # -- Defence-in-depth budget guard --
    ceiling   = int(os.environ.get("TOKEN_BUDGET_LIMIT", 10_000))
    spent     = state.get("token_spend", 0)
    remaining = ceiling - spent
    reserve   = _REPAIR_RESERVE if repair_mode else _FULL_GEN_RESERVE

    if remaining < reserve:
        halt_error = (
            f"BUDGET HALT: {remaining:,} tokens remaining, "
            f"{reserve:,} required for {'repair' if repair_mode else 'generation'} pass. "
            f"Raise TOKEN_BUDGET_LIMIT in .env (current ceiling: {ceiling:,})."
        )
        halt_span = {
            "agent": "StoryGeneratorAgent[budget-halt]",
            "duration_ms": 0, "tokens": 0,
            "repair_mode": repair_mode,
            "tokens_used": spent, "token_ceiling": ceiling,
            "cycles": (state.get("validation") or {}).get("cycles_detected", 0),
            "success": False, "error": halt_error,
        }
        return {
            **state,
            "story":       last_story,
            "story_json":  __import__("json").dumps(last_story) if last_story else None,
            "approved":    False,
            "budget_halt": True,
            "repair_mode": repair_mode,
            "retry_count": state.get("max_retries", 2),
            "agent_spans": [*state.get("agent_spans", []), halt_span],
            "error":       halt_error,
        }

    # -- Main generation / repair path --
    if repair_mode:
        validation_issues = (state.get("validation") or {}).get("issues") or []
        user_prompt = _build_repair_prompt(last_story, broken_nodes, validation_issues)
        try:
            raw, tokens = _call_groq(REPAIR_SYSTEM_PROMPT, user_prompt, max_tokens=1024)
            patch = _parse_json(raw)
            if not isinstance(patch, dict):
                raise ValueError("Patch is not a JSON object")
            story = _break_cycles(merge_patch(last_story, patch))
        except Exception as groq_err:
            error = f"Groq repair: {groq_err}"
            try:
                raw, tokens = _call_gemini(REPAIR_SYSTEM_PROMPT, user_prompt, max_tokens=1024)
                patch = _parse_json(raw)
                story = _break_cycles(merge_patch(last_story, patch))
                error = None
            except Exception as gemini_err:
                error = f"Both LLMs failed in repair mode. Groq: {groq_err} | Gemini: {gemini_err}"
        agent_label = "StoryGeneratorAgent[repair]"
    else:
        try:
            raw, tokens = _call_groq(SYSTEM_PROMPT, _build_user_prompt(state["premise"], state["genre"], state["tone"]))
            story = _break_cycles(_parse_json(raw))
        except Exception as groq_err:
            error = f"Groq: {groq_err}"
            try:
                raw, tokens = _call_gemini(SYSTEM_PROMPT, _build_user_prompt(state["premise"], state["genre"], state["tone"]))
                story = _break_cycles(_parse_json(raw))
                error = None
            except Exception as gemini_err:
                groq_msg, gemini_msg = str(groq_err), str(gemini_err)
                if "rate_limit_exceeded" in groq_msg or "429" in groq_msg:
                    if "rate_limit_exceeded" in gemini_msg or "429" in gemini_msg or "quota" in gemini_msg.lower():
                        error = (
                            "API quota exhausted on both providers (Groq + Gemini). "
                            "Groq daily limit resets every 24 hours; Gemini free tier resets daily."
                        )
                    else:
                        error = f"Both LLMs failed. Groq: {groq_err} | Gemini: {gemini_err}"
                else:
                    error = f"Both LLMs failed. Groq: {groq_err} | Gemini: {gemini_err}"
        agent_label = "StoryGeneratorAgent"

    span = {
        "agent":         agent_label,
        "duration_ms":   int((time.time() - span_start) * 1000),
        "tokens":        tokens,
        "repair_mode":   repair_mode,
        "tokens_used":   spent + tokens,
        "token_ceiling": ceiling,
        "cycles":        (state.get("validation") or {}).get("cycles_detected", 0),
        "success":       story is not None,
        "error":         error,
    }

    if story is None:
        return {
            **state,
            "story":       None, "story_json": None,
            "approved":    False, "repair_mode": repair_mode,
            "retry_count": state.get("max_retries", 2),
            "token_spend": spent + tokens,
            "agent_spans": [*state.get("agent_spans", []), span],
            "error":       error,
        }

    return {
        **state,
        "story":       story,
        "story_json":  __import__("json").dumps(story),
        "repair_mode": repair_mode,
        "token_spend": spent + tokens,
        "agent_spans": [*state.get("agent_spans", []), span],
        "error":       error,
    }

In [ ]:
%%writefile agents/style_vault_agent.py
"""
StyleVaultAgent — FAISS + sentence-transformers style gate.
"""
from __future__ import annotations

import os
import time
from typing import Any

from agents.state import ScenePilotState

_vault = None


def _get_vault():
    global _vault
    if _vault is None:
        from core.style_vault import StyleVault
        _vault = StyleVault()
        rules_dir = os.path.join(os.path.dirname(__file__), "..", "data", "rules")
        _vault.load_rules_dir(os.path.abspath(rules_dir))
    return _vault


VALID_TONES_BY_GENRE: dict[str, set[str]] = {
    "thriller":    {"tense", "dark", "neutral"},
    "fantasy":     {"hopeful", "tense", "neutral", "dark"},
    "sci-fi":      {"tense", "neutral", "dark", "hopeful"},
    "educational": {"neutral", "hopeful", "playful"},
    "marketing":   {"hopeful", "playful", "neutral"},
}


def _check_tone_consistency(story: dict[str, Any], genre: str) -> list[str]:
    allowed    = VALID_TONES_BY_GENRE.get(genre, set())
    violations: list[str] = []
    for scene in story.get("scenes", []):
        tone = scene.get("tone", "neutral")
        if allowed and tone not in allowed:
            violations.append(
                f"Scene {scene.get('id', '?')} has tone '{tone}' "
                f"which is not suitable for '{genre}' genre."
            )
    return violations


def _check_with_faiss(story: dict[str, Any]) -> list[str]:
    vault      = _get_vault()
    violations: list[str] = []
    threshold  = float(os.environ.get("STYLE_SIMILARITY_THRESHOLD", "0.35"))
    for scene in story.get("scenes", []):
        text = scene.get("text", "")
        if not text:
            continue
        results = vault.query(text, k=1)
        if results:
            score, rule_text = results[0]
            if score < threshold:
                violations.append(
                    f"Scene {scene.get('id', '?')} may violate style guidelines "
                    f"(similarity={score:.3f}). Nearest rule: \"{rule_text[:80]}...\""
                )
    return violations


def style_vault_node(state: ScenePilotState) -> ScenePilotState:
    span_start = time.time()
    violations: list[str] = []

    story = state.get("story")
    if story is None:
        span = {"agent": "StyleVaultAgent", "duration_ms": 0, "violations": 0, "success": False}
        return {**state, "style_check": {"violations": []},
                "agent_spans": [*state.get("agent_spans", []), span]}

    try:
        violations.extend(_check_tone_consistency(story, state.get("genre", "")))
        violations.extend(_check_with_faiss(story))
    except Exception as exc:
        violations.append(f"StyleVault error (non-blocking): {exc}")

    span = {
        "agent":       "StyleVaultAgent",
        "duration_ms": int((time.time() - span_start) * 1000),
        "violations":  len(violations),
        "success":     True,
    }

    current_validation = state.get("validation") or {
        "passed": True, "issues": [], "cycles_detected": 0,
        "schema_errors": [], "style_violations": [],
    }

    return {
        **state,
        "style_check": {"violations": violations},
        "validation":  {**current_validation, "style_violations": violations},
        "agent_spans": [*state.get("agent_spans", []), span],
    }

In [ ]:
%%writefile agents/sandbox_validator.py
"""
SandboxValidatorAgent — runs story JSON through cycle detection and schema
validation. Extracts invalid_edges (cycle-forming back-edges) and writes
broken_nodes + last_story to state for diff-patch repair.
"""
from __future__ import annotations

import time

from agents.state import ScenePilotState
from sandbox.validator import validate_story
from core.telemetry import (
    STORIES_GENERATED, LOOP_DETECTIONS, SANDBOX_REJECTIONS, VALIDATION_DURATION,
)


def sandbox_validator_node(state: ScenePilotState) -> ScenePilotState:
    span_start = time.time()

    story = state.get("story")
    if story is None:
        span = {"agent": "SandboxValidatorAgent", "duration_ms": 0, "passed": False, "success": False}
        return {**state, "approved": False, "agent_spans": [*state.get("agent_spans", []), span]}

    result   = validate_story(story)
    duration = time.time() - span_start
    VALIDATION_DURATION.observe(duration)

    if result["cycles_detected"] > 0:
        LOOP_DETECTIONS.inc(result["cycles_detected"])

    passed   = result["passed"]
    approved = passed and len(state.get("style_check", {}).get("violations", [])) == 0

    if not approved:
        SANDBOX_REJECTIONS.inc()
    else:
        STORIES_GENERATED.inc()

    # Extract cycle-forming edge pairs for targeted repair
    invalid_edges: list[tuple[str, str]] = result.get("invalid_edges", [])
    broken_nodes = invalid_edges if invalid_edges else state.get("broken_nodes") or []
    last_story   = story if not approved else state.get("last_story")

    span = {
        "agent":        "SandboxValidatorAgent",
        "duration_ms":  int(duration * 1000),
        "passed":       passed,
        "cycles":       result["cycles_detected"],
        "invalid_edges": invalid_edges,
        "schema_errors": len(result["schema_errors"]),
        "success":      True,
    }

    current_validation = state.get("validation") or {
        "passed": True, "issues": [], "cycles_detected": 0,
        "schema_errors": [], "style_violations": [],
    }
    merged_validation = {
        **current_validation,
        "passed":           passed,
        "cycles_detected":  result["cycles_detected"],
        "schema_errors":    result["schema_errors"],
        "issues":           result["schema_errors"] + current_validation.get("style_violations", []),
    }

    return {
        **state,
        "validation":   merged_validation,
        "approved":     approved,
        "broken_nodes": broken_nodes,
        "last_story":   last_story,
        "agent_spans":  [*state.get("agent_spans", []), span],
    }

In [ ]:
%%writefile agents/compliance_agent.py
"""
ComplianceAgent — SHA-256 fingerprints the final story and writes a
structured audit entry. Runs even on failure so the audit trail is never lost.
"""
from __future__ import annotations

import hashlib
import json
import time
from datetime import datetime, timezone

from agents.state import AuditEntry, ScenePilotState
from core.story_store import story_store


def compliance_node(state: ScenePilotState) -> ScenePilotState:
    span_start = time.time()

    story    = state.get("story")
    story_id = state.get("story_id", "unknown")

    fingerprint = ""
    if story:
        raw         = json.dumps(story, sort_keys=True, ensure_ascii=False)
        fingerprint = hashlib.sha256(raw.encode()).hexdigest()

    audit: AuditEntry = {
        "story_id":    story_id,
        "fingerprint": fingerprint,
        "timestamp":   datetime.now(timezone.utc).isoformat(),
        "agent_spans": state.get("agent_spans", []),
        "token_spend": state.get("token_spend", 0),
        "validation":  state.get("validation") or {
            "passed": False, "issues": [],
            "cycles_detected": 0, "schema_errors": [], "style_violations": [],
        },
    }

    story_store.save(story_id, {
        "story":    story,
        "audit":    audit,
        "approved": state.get("approved", False),
        "premise":  state.get("premise", ""),
        "genre":    state.get("genre", ""),
        "tone":     state.get("tone", 0.5),
    })

    span = {
        "agent":       "ComplianceAgent",
        "duration_ms": int((time.time() - span_start) * 1000),
        "fingerprint": fingerprint[:16] + "...",
        "success":     True,
    }

    return {
        **state,
        "audit":       audit,
        "agent_spans": [*state.get("agent_spans", []), span],
    }

In [ ]:
%%writefile agents/orchestrator.py
"""
Orchestrator — LangGraph StateGraph with budget-aware retry router.

Graph topology:
  generate -> style_vault -> sandbox
                                 |
               approved ----------+
                  |   retry < max AND _has_budget_for_retry() -> retry -> generate
                  |   else -> fail
                  v           |
             compliance <------
                  |
                 END
"""
from __future__ import annotations

import os
import uuid
from typing import Literal

from langgraph.graph import END, StateGraph  # type: ignore

from agents.state import ScenePilotState
from agents.story_generator import story_generator_node
from agents.style_vault_agent import style_vault_node
from agents.sandbox_validator import sandbox_validator_node
from agents.compliance_agent import compliance_node

# Reserve constants — must mirror agents/story_generator.py
_REPAIR_RESERVE: int   = 1_200
_FULL_GEN_RESERVE: int = 9_500


def _has_budget_for_retry(state: ScenePilotState) -> bool:
    """Return True only when sufficient token headroom remains for one more LLM pass.
    Reads ceiling live from env — never caches in state to avoid stale-value bugs.
    """
    ceiling   = int(os.environ.get("TOKEN_BUDGET_LIMIT", 10_000))
    remaining = ceiling - state.get("token_spend", 0)
    will_repair = bool(state.get("broken_nodes")) and state.get("last_story") is not None
    reserve = _REPAIR_RESERVE if will_repair else _FULL_GEN_RESERVE
    return remaining >= reserve


def _route_after_sandbox(state: ScenePilotState) -> Literal["compliance", "retry", "fail"]:
    if state.get("approved"):
        return "compliance"
    can_retry = (
        state.get("retry_count", 0) < state.get("max_retries", 2)
        and _has_budget_for_retry(state)
    )
    return "retry" if can_retry else "fail"


def _increment_retry(state: ScenePilotState) -> ScenePilotState:
    """Bump retry counter. Preserve last_story + broken_nodes for repair mode."""
    return {
        **state,
        "retry_count": state.get("retry_count", 0) + 1,
        "story":       None,
        "story_json":  None,
        "approved":    False,
        "validation":  None,
        "style_check": None,
        "repair_mode": False,  # generator sets this on next pass
    }


def _fail_node(state: ScenePilotState) -> ScenePilotState:
    """Terminal failure. Detects budget-gate halt vs retry exhaustion."""
    budget_halt: bool = state.get("budget_halt", False)
    if not budget_halt and state.get("retry_count", 0) < state.get("max_retries", 2):
        budget_halt = True  # reached fail before exhausting retries => budget gate fired
    default_error = (
        "Insufficient token budget for next retry. Raise TOKEN_BUDGET_LIMIT in .env."
        if budget_halt else "Max retries exceeded."
    )
    return {
        **state,
        "approved":    False,
        "budget_halt": budget_halt,
        "error":       state.get("error") or default_error,
    }


def build_graph() -> StateGraph:
    graph = StateGraph(ScenePilotState)

    graph.add_node("generate",   story_generator_node)
    graph.add_node("style_vault", style_vault_node)
    graph.add_node("sandbox",    sandbox_validator_node)
    graph.add_node("compliance", compliance_node)
    graph.add_node("retry",      _increment_retry)
    graph.add_node("fail",       _fail_node)

    graph.set_entry_point("generate")
    graph.add_edge("generate",   "style_vault")
    graph.add_edge("style_vault", "sandbox")
    graph.add_conditional_edges(
        "sandbox", _route_after_sandbox,
        {"compliance": "compliance", "retry": "retry", "fail": "fail"},
    )
    graph.add_edge("retry",      "generate")
    graph.add_edge("compliance", END)
    graph.add_edge("fail",       "compliance")

    return graph


_compiled = None


def get_compiled_graph():
    global _compiled
    if _compiled is None:
        _compiled = build_graph().compile()
    return _compiled


def run_pipeline(premise: str, genre: str, tone: float) -> ScenePilotState:
    story_id = str(uuid.uuid4())
    initial: ScenePilotState = {
        "story_id":    story_id,
        "premise":     premise,
        "genre":       genre,
        "tone":        tone,
        "story":       None,
        "story_json":  None,
        "validation":  None,
        "style_check": None,
        "approved":    False,
        "retry_count": 0,
        "max_retries": int(os.environ.get("MAX_RETRIES", 2)),
        "broken_nodes": None,
        "last_story":   None,
        "repair_mode":  False,
        "budget_halt":  False,
        "audit":       None,
        "agent_spans": [],
        "token_spend": 0,
        "error":       None,
    }
    return get_compiled_graph().invoke(initial)

---
## 🔬 SECTION 4 — Sandbox Validator + invalid_edges Extraction

In [ ]:
%%writefile sandbox/validator.py
"""
Core story validation logic.

Three independent check passes:
  1. _check_schema()    -- required fields, valid tones, duplicate IDs
  2. _check_cycles()    -- networkx simple_cycles(); extracts invalid_edges
  3. _check_structure() -- dead-end refs, orphaned scenes, min scene count

invalid_edges: the exact (source_id, target_id) back-edge pairs that form
cycles. Passed to SandboxValidatorAgent -> state.broken_nodes -> repair prompt.
"""
from __future__ import annotations

from typing import Any

REQUIRED_SCENE_KEYS = {"id", "text", "tone", "choices"}
VALID_TONES         = {"tense", "hopeful", "dark", "neutral", "playful"}
MIN_SCENES          = 6


def validate_story(story: dict[str, Any]) -> dict[str, Any]:
    schema_errors = _check_schema(story)
    cycles, cycle_issues, invalid_edges = _check_cycles(story)
    structural    = _check_structure(story)

    issues = schema_errors + cycle_issues + structural
    passed = len(issues) == 0

    return {
        "passed":              passed,
        "issues":              issues,
        "cycles_detected":     cycles,
        "invalid_edges":       invalid_edges,
        "schema_errors":       schema_errors,
        "structural_warnings": structural,
    }


def _check_schema(story: dict[str, Any]) -> list[str]:
    errors: list[str] = []
    if not isinstance(story, dict):
        return ["Story must be a JSON object."]
    if "title" not in story:
        errors.append("Missing required field: 'title'.")
    scenes = story.get("scenes")
    if not scenes or not isinstance(scenes, list):
        errors.append("Missing or empty 'scenes' array.")
        return errors
    scene_ids: set[str] = set()
    for i, scene in enumerate(scenes):
        prefix = f"Scene[{i}]"
        if not isinstance(scene, dict):
            errors.append(f"{prefix} is not an object.")
            continue
        for key in REQUIRED_SCENE_KEYS:
            if key not in scene:
                errors.append(f"{prefix} missing field '{key}'.")
        sid = scene.get("id")
        if sid:
            if sid in scene_ids:
                errors.append(f"Duplicate scene id '{sid}'.")
            scene_ids.add(str(sid))
        tone = scene.get("tone", "neutral")
        if tone not in VALID_TONES:
            errors.append(f"{prefix} ({sid}) has invalid tone '{tone}'. Allowed: {sorted(VALID_TONES)}.")
        choices = scene.get("choices", [])
        if not isinstance(choices, list):
            errors.append(f"{prefix} ({sid}) 'choices' must be an array.")
        else:
            for j, choice in enumerate(choices):
                if "text" not in choice:
                    errors.append(f"{prefix} choice[{j}] missing 'text'.")
    return errors


def _check_cycles(story: dict[str, Any]) -> tuple[int, list[str], list[tuple[str, str]]]:
    """Return (cycle_count, human_issues, invalid_edges).

    invalid_edges: deduplicated list of (src, dst) pairs that are
    back-edges closing cycles -- the exact routing properties to patch.
    """
    try:
        import networkx as nx  # type: ignore
    except ImportError:
        return 0, [], []

    G = nx.DiGraph()
    for scene in story.get("scenes", []):
        sid = scene.get("id")
        if not sid:
            continue
        G.add_node(sid)
        for choice in scene.get("choices", []):
            nxt = choice.get("next")
            if nxt:
                G.add_edge(sid, nxt)

    cycles = list(nx.simple_cycles(G))
    issues = [f"Cycle detected: {' -> '.join(c + [c[0]])}" for c in cycles]

    seen: set[tuple[str, str]] = set()
    invalid_edges: list[tuple[str, str]] = []
    for cycle in cycles:
        for i, node in enumerate(cycle):
            edge = (node, cycle[(i + 1) % len(cycle)])
            if edge not in seen:
                seen.add(edge)
                invalid_edges.append(edge)

    return len(cycles), issues, invalid_edges


def _check_structure(story: dict[str, Any]) -> list[str]:
    issues: list[str] = []
    scenes = story.get("scenes", [])
    if not scenes:
        return issues
    if len(scenes) < MIN_SCENES:
        issues.append(f"Story has only {len(scenes)} scenes (minimum required: {MIN_SCENES}).")
    scene_ids  = {s.get("id") for s in scenes if s.get("id")}
    for scene in scenes:
        sid     = scene.get("id")
        choices = scene.get("choices", []) if isinstance(scene.get("choices"), list) else []
        for choice in choices:
            nxt = choice.get("next")
            if nxt:
                if nxt not in scene_ids:
                    issues.append(
                        f"Scene '{sid}' choice '{choice.get('text', '?')}' "
                        f"points to non-existent scene '{nxt}'."
                    )
    if scenes:
        root      = scenes[0].get("id")
        scene_map = {s.get("id"): s for s in scenes if s.get("id")}
        visited: set[str] = set()
        queue = [root]
        while queue:
            nid = queue.pop()
            if nid in visited or nid not in scene_map:
                continue
            visited.add(nid)
            for choice in scene_map[nid].get("choices", []):
                nxt = choice.get("next")
                if nxt:
                    queue.append(nxt)
        for oid in sorted(scene_ids - visited):
            issues.append(f"Scene '{oid}' is unreachable from the root scene.")
    return issues


if __name__ == "__main__":
    import json, sys
    data   = json.loads(sys.stdin.read())
    result = validate_story(data)
    print(json.dumps(result))
    sys.exit(0 if result["passed"] else 1)

In [ ]:
%%writefile sandbox/runner.py
"""
Sandbox runner — Docker-isolated execution first;
falls back to in-process when SANDBOX_USE_DOCKER=false or Docker unavailable.
"""
from __future__ import annotations

import json
import os
import subprocess
import sys
import tempfile
from typing import Any

from sandbox.validator import validate_story as _inprocess_validate

DOCKER_IMAGE = os.environ.get("SANDBOX_DOCKER_IMAGE", "python:3.11-slim")
USE_DOCKER   = os.environ.get("SANDBOX_USE_DOCKER", "false").lower() == "true"


def run_in_docker(story_json: str) -> dict[str, Any]:
    validator_path = os.path.join(os.path.dirname(__file__), "validator.py")
    with tempfile.NamedTemporaryFile(suffix=".py", mode="w", delete=False, encoding="utf-8") as tmp:
        tmp.write(open(validator_path, encoding="utf-8").read())
        tmp_path = tmp.name
    try:
        result = subprocess.run(
            ["docker", "run", "--rm", "--network", "none",
             "--memory", "128m", "--cpus", "0.5", "-i",
             "-v", f"{tmp_path}:/validator.py:ro",
             DOCKER_IMAGE, "python", "/validator.py"],
            input=story_json, capture_output=True, text=True, timeout=30,
        )
        if result.returncode not in (0, 1):
            raise RuntimeError(f"Docker exited {result.returncode}: {result.stderr}")
        return json.loads(result.stdout)
    except (subprocess.TimeoutExpired, FileNotFoundError, RuntimeError) as exc:
        raise RuntimeError(f"Docker sandbox failed: {exc}") from exc
    finally:
        os.unlink(tmp_path)


def validate_story(story: dict[str, Any]) -> dict[str, Any]:
    if USE_DOCKER:
        try:
            return run_in_docker(json.dumps(story))
        except Exception as exc:
            print(f"[sandbox] Docker run failed ({exc}), using in-process fallback.", file=sys.stderr)
    return _inprocess_validate(story)

---
## 🔌 SECTION 5 — FastAPI Gateway + Pre-flight Budget Estimator

In [ ]:
%%writefile api/routes.py
"""
Core story API routes.

POST /api/generate   -> run_pipeline()   [3-layer budget gate]
POST /api/validate   -> sandbox only, no LLM
POST /api/blueprint  -> 3D spatial transform
GET  /api/stories    -> list stored IDs
GET  /api/stories/{id} -> retrieve story
GET  /api/audit/{id}   -> retrieve audit
"""
from __future__ import annotations

import os
import uuid
from typing import Any, Optional

from fastapi import APIRouter, HTTPException
from pydantic import BaseModel, Field

from agents.orchestrator import run_pipeline
from core.story_store import story_store
from core.telemetry import STYLE_VIOLATIONS

router = APIRouter(prefix="/api", tags=["stories"])


def _get_ceiling() -> int:
    return int(os.environ.get("TOKEN_BUDGET_LIMIT", 10_000))


def _estimate_run_cost(premise: str, max_retries: int) -> dict[str, int]:
    """Model worst-case token budget for a full pipeline run.

    full_gen_est  = 400 (system overhead) + premise_tokens * SCENE_MULTIPLIER
    repair_reserve = 1,200 * MAX_RETRIES
    worst_case     = full_gen_est + repair_reserve
    recommended    = worst_case * 1.15, rounded up to nearest 500

    SCENE_MULTIPLIER default 40 is calibrated to complex thriller/sci-fi premises.
    Lower to 15-20 for educational/marketing.
    """
    premise_tokens:   int = max(len(premise) // 4, 1)
    scene_multiplier: int = int(os.environ.get("SCENE_MULTIPLIER", 40))
    full_gen_est:     int = 400 + premise_tokens * scene_multiplier
    repair_reserve:   int = 1_200 * max_retries
    worst_case:       int = full_gen_est + repair_reserve
    recommended:      int = ((int(worst_case * 1.15) + 499) // 500) * 500
    return {
        "premise_tokens":    premise_tokens,
        "full_gen_est":      full_gen_est,
        "repair_reserve":    repair_reserve,
        "worst_case":        worst_case,
        "recommended_ceiling": recommended,
    }


class GenerateRequest(BaseModel):
    premise: str   = Field(..., min_length=10, max_length=2000)
    genre:   str   = Field("thriller", pattern=r"^(thriller|fantasy|sci-fi|educational|marketing)$")
    tone:    float = Field(0.5, ge=0.0, le=1.0)


class GenerateResponse(BaseModel):
    story_id:      str
    approved:      bool
    title:         Optional[str]
    scenes:        Optional[list[dict[str, Any]]]
    validation:    Optional[dict[str, Any]]
    agent_spans:   list[dict[str, Any]]
    token_spend:   int
    token_ceiling: int
    error:         Optional[str]


@router.post("/generate", response_model=GenerateResponse)
def generate_story(req: GenerateRequest):
    ceiling     = _get_ceiling()
    max_retries = int(os.environ.get("MAX_RETRIES", 2))
    estimate    = _estimate_run_cost(req.premise, max_retries)

    # Layer 1: pre-flight gate
    if estimate["worst_case"] > ceiling:
        return GenerateResponse(
            story_id=str(uuid.uuid4()), approved=False,
            title=None, scenes=None, validation=None, agent_spans=[],
            token_spend=0, token_ceiling=ceiling,
            error=(
                f"PRE-FLIGHT REJECTED: This premise requires an estimated "
                f"~{estimate['full_gen_est']:,} tokens to generate "
                f"plus {estimate['repair_reserve']:,} tokens for up to {max_retries} repair "
                f"{'retry' if max_retries == 1 else 'retries'} "
                f"(~{estimate['worst_case']:,} total). "
                f"Your current ceiling is {ceiling:,}. "
                f"Set TOKEN_BUDGET_LIMIT={estimate['recommended_ceiling']:,} in .env."
            ),
        )

    state = run_pipeline(premise=req.premise, genre=req.genre, tone=req.tone)

    sv = (state.get("validation") or {}).get("style_violations", [])
    if sv:
        STYLE_VIOLATIONS.inc(len(sv))

    spent       = state.get("token_spend", 0)
    budget_halt = state.get("budget_halt", False)
    error       = state.get("error")

    if error:
        if "Unterminated string" in error or "JSONDecodeError" in error or "Expecting value" in error:
            error = (
                "Generation Truncated -- The LLM response was cut off before the JSON "
                "structure could be completed. Please retry."
            )

    budget_exceeded = spent > ceiling
    if not error and budget_exceeded and not budget_halt:
        cycles = (state.get("validation") or {}).get("cycles_detected", 0)
        error = (
            f"BUDGET EXHAUSTED: {spent:,} tokens consumed (ceiling: {ceiling:,}, "
            f"overage: {spent - ceiling:,}). "
            + (f"Cycle repairs attempted: {cycles}. " if cycles else "")
            + f"Set TOKEN_BUDGET_LIMIT={estimate['recommended_ceiling']:,} in .env."
        )

    approved = state.get("approved", False) and not budget_exceeded and not budget_halt

    story = state.get("story") or {}
    return GenerateResponse(
        story_id      = state["story_id"],
        approved      = approved,
        title         = story.get("title"),
        scenes        = story.get("scenes"),
        validation    = state.get("validation"),
        agent_spans   = state.get("agent_spans", []),
        token_spend   = spent,
        token_ceiling = ceiling,
        error         = error,
    )


@router.get("/stories/{story_id}")
def get_story(story_id: str):
    record = story_store.get(story_id)
    if not record:
        raise HTTPException(status_code=404, detail="Story not found.")
    return record


@router.get("/audit/{story_id}")
def get_audit(story_id: str):
    record = story_store.get(story_id)
    if not record or "audit" not in record:
        raise HTTPException(status_code=404, detail="Audit not found.")
    return record["audit"]


@router.get("/stories")
def list_stories():
    return {"story_ids": story_store.all_ids()}


@router.post("/blueprint")
def generate_blueprint_endpoint(body: dict):
    from core.blueprint import generate_blueprint
    return generate_blueprint(body.get("story", body), story_id=body.get("story_id", "story"))


@router.post("/validate")
def validate_story_endpoint(body: dict):
    from sandbox.validator import validate_story
    return validate_story(body.get("story", body))

In [ ]:
%%writefile api/main.py
"""
FastAPI application entrypoint.
"""
from __future__ import annotations

import os

from fastapi import FastAPI, Response
from fastapi.middleware.cors import CORSMiddleware
from prometheus_client import CONTENT_TYPE_LATEST, generate_latest

from core.telemetry import setup_otel
from api.routes import router as story_router
from api.samples_route import router as samples_router

setup_otel()

app = FastAPI(
    title       = "ScenePilot AI",
    version     = "2.0.0",
    description = "AI-powered branching narrative generator with diff-patch repair and token budget gates.",
)

origins = os.environ.get(
    "CORS_ORIGINS", "http://localhost:5173,http://localhost:3000"
).split(",")

app.add_middleware(
    CORSMiddleware,
    allow_origins=origins, allow_credentials=True,
    allow_methods=["*"], allow_headers=["*"],
)

app.include_router(story_router)
app.include_router(samples_router)


@app.get("/metrics", include_in_schema=False)
def metrics():
    return Response(content=generate_latest(), media_type=CONTENT_TYPE_LATEST)


@app.get("/health")
def health():
    return {"status": "ok", "service": "scenepilot-ai", "version": "2.0.0"}

---
## 🐳 SECTION 6 — Docker Infrastructure & Deployment

In [ ]:
%%writefile Dockerfile
FROM python:3.11-slim

WORKDIR /app

RUN apt-get update && apt-get install -y --no-install-recommends \
    build-essential \
    && rm -rf /var/lib/apt/lists/*

COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt

COPY . .

RUN adduser --disabled-password --gecos "" appuser \
    && chown -R appuser:appuser /app
USER appuser

EXPOSE 8000

CMD ["uvicorn", "api.main:app", "--host", "0.0.0.0", "--port", "8000"]

In [ ]:
%%writefile docker-compose.yml
services:
  api:
    build: { context: ., dockerfile: Dockerfile }
    ports: ["8000:8000"]
    env_file: [.env]
    environment:
      - PYTHONUNBUFFERED=1
      - SANDBOX_USE_DOCKER=false
    volumes:
      - /var/run/docker.sock:/var/run/docker.sock
    depends_on: [prometheus]
    restart: unless-stopped

  frontend:
    build: { context: ./frontend, dockerfile: Dockerfile }
    ports: ["5173:80"]
    depends_on: [api]
    restart: unless-stopped

  prometheus:
    image: prom/prometheus:latest
    ports: ["9090:9090"]
    volumes:
      - ./prometheus/prometheus.yml:/etc/prometheus/prometheus.yml:ro
    command:
      - "--config.file=/etc/prometheus/prometheus.yml"
      - "--storage.tsdb.path=/prometheus"
    restart: unless-stopped

  grafana:
    image: grafana/grafana:latest
    ports: ["3001:3000"]
    environment:
      - GF_SECURITY_ADMIN_PASSWORD=scenepilot
    volumes:
      - grafana_data:/var/lib/grafana
    depends_on: [prometheus]
    restart: unless-stopped

volumes:
  grafana_data:

In [ ]:
# Cell 20 — Build and start all 4 services in detached mode
#
# First build: ~3-5 min (downloads Python 3.11-slim + pip installs)
# Subsequent builds: ~30s (layer cache)
#
# Services started:
#   api        -> http://localhost:8000
#   frontend   -> http://localhost:5173
#   prometheus -> http://localhost:9090
#   grafana    -> http://localhost:3001  (admin / scenepilot)

!docker compose up --build -d

import time, subprocess, urllib.request, json
print("Waiting 8s for containers to initialise...")
time.sleep(8)

# Container status
result = subprocess.run(
    ["docker", "compose", "ps", "--format", "table {{.Name}}\t{{.Status}}\t{{.Ports}}"],
    capture_output=True, text=True,
)
print(result.stdout)

# Health probe
try:
    with urllib.request.urlopen("http://localhost:8000/health", timeout=8) as r:
        body = json.loads(r.read())
    print(f"  Health: {body}")
except Exception as e:
    print(f"  Health check failed: {e}")

# Registered routes
try:
    with urllib.request.urlopen("http://localhost:8000/openapi.json", timeout=8) as r:
        spec = json.loads(r.read())
    print("  Routes:", sorted(spec["paths"].keys()))
except Exception as e:
    print(f"  OpenAPI spec unavailable: {e}")

# Prometheus metrics
try:
    with urllib.request.urlopen("http://localhost:8000/metrics", timeout=8) as r:
        raw = r.read().decode()
    sp = [l for l in raw.splitlines() if l.startswith("scenepilot_") and not l.startswith("#")]
    print("  ScenePilot metrics:")
    for m in sp[:7]:
        print(f"    {m}")
except Exception as e:
    print(f"  Metrics endpoint failed: {e}")

---
## 🧪 SECTION 7 — End-to-End Test Suite

Four test blocks run entirely **in-process** — no LLM keys required, no Docker, no HTTP.

```
Test A  -- sandbox/validator.py: schema, cycles, invalid_edges, structural checks
Test B  -- core/utils.py: merge_patch correctness
Test C  -- orchestrator budget-gate + retry-loop state machine simulation
Test D  -- Prometheus counters + StoryStore CRUD
```

In [ ]:
# Cell 21 — Test A: sandbox/validator.py + invalid_edges extraction
import sys, os
sys.path.insert(0, os.getcwd())

from sandbox.validator import validate_story

PASS = FAIL = 0

def chk(label, cond, detail=""):
    global PASS, FAIL
    if cond:
        print(f"  OK   {label}")
        PASS += 1
    else:
        print(f"  FAIL {label}  {detail}")
        FAIL += 1

print("=" * 60)
print("TEST A -- sandbox/validator.py")
print("=" * 60)

# A1: Missing title
r = validate_story({"scenes": [{"id":"s1","text":"x","tone":"tense","choices":[]}]})
chk("A1 missing title detected", any("title" in e for e in r["schema_errors"]), str(r["schema_errors"]))

# A2: Invalid tone
r = validate_story({"title":"T","scenes":[{"id":"s1","text":"x","tone":"INVALID","choices":[]}]})
chk("A2 invalid tone detected", any("INVALID" in e for e in r["schema_errors"]), str(r["schema_errors"]))

# A3: Cycle detected + invalid_edges populated
cyclic = {
    "title": "Cyclic",
    "scenes": [
        {"id":"s1","text":"Start","tone":"tense",  "choices":[{"text":"go",  "next":"s2"}]},
        {"id":"s2","text":"Mid",  "tone":"dark",   "choices":[{"text":"back","next":"s1"}]},
        {"id":"s3","text":"E1",   "tone":"tense",  "choices":[]},
        {"id":"s4","text":"E2",   "tone":"dark",   "choices":[]},
        {"id":"s5","text":"E3",   "tone":"tense",  "choices":[]},
        {"id":"s6","text":"E4",   "tone":"tense",  "choices":[]},
    ]
}
r = validate_story(cyclic)
chk("A3 cycle s1->s2->s1 detected",   r["cycles_detected"] >= 1, f"cycles={r['cycles_detected']}")
chk("A3 invalid_edges populated",     len(r["invalid_edges"]) >= 1, str(r["invalid_edges"]))
chk("A3 invalid_edges are tuples",    all(len(e) == 2 for e in r["invalid_edges"]), str(r["invalid_edges"]))
chk("A3 story marked failed",         r["passed"] is False)

# A4: Dangling reference
dangling = {
    "title": "Dangling",
    "scenes": [
        {"id":"s1","text":"Start","tone":"tense","choices":[{"text":"go","next":"GHOST"}]},
        {"id":"s2","text":"E1",  "tone":"tense","choices":[]},
        {"id":"s3","text":"E2",  "tone":"tense","choices":[]},
        {"id":"s4","text":"E3",  "tone":"dark", "choices":[]},
        {"id":"s5","text":"E4",  "tone":"dark", "choices":[]},
        {"id":"s6","text":"E5",  "tone":"dark", "choices":[]},
    ]
}
r = validate_story(dangling)
chk("A4 dangling ref 'GHOST' detected", any("GHOST" in w for w in r["structural_warnings"]), str(r["structural_warnings"]))

# A5: Too few scenes
r = validate_story({"title":"Tiny","scenes":[
    {"id":"s1","text":"A","tone":"tense","choices":[{"text":"go","next":"s2"}]},
    {"id":"s2","text":"B","tone":"dark", "choices":[]},
]})
chk("A5 min scene count (2 < 6) rejected", any("minimum" in w for w in r["structural_warnings"]), str(r["structural_warnings"]))

# A6: Clean story
clean = {
    "title": "Good Story",
    "scenes": [
        {"id":"s1","text":"Open",   "tone":"tense",  "choices":[{"text":"A","next":"s2"},{"text":"B","next":"s3"}]},
        {"id":"s2","text":"Path A", "tone":"dark",   "choices":[{"text":"C","next":"s4"},{"text":"D","next":"s5"}]},
        {"id":"s3","text":"Path B", "tone":"hopeful","choices":[{"text":"E","next":"s6"}]},
        {"id":"s4","text":"End 1",  "tone":"dark",   "choices":[]},
        {"id":"s5","text":"End 2",  "tone":"tense",  "choices":[]},
        {"id":"s6","text":"End 3",  "tone":"hopeful","choices":[]},
    ]
}
r = validate_story(clean)
chk("A6 clean story passes",      r["passed"] is True, str(r["issues"]))
chk("A6 zero cycles",             r["cycles_detected"] == 0)
chk("A6 no invalid_edges",        r["invalid_edges"] == [])
chk("A6 no schema errors",        r["schema_errors"] == [])

In [ ]:
# Cell 22 — Test B: core/utils.py merge_patch correctness
from core.utils import merge_patch

print("=" * 60)
print("TEST B -- core/utils.py merge_patch")
print("=" * 60)

original = {
    "title": "Original",
    "scenes": [
        {"id": "s1", "text": "Start", "tone": "tense",
         "choices": [{"text": "old", "next": "s1"}]},   # broken (back-edge)
        {"id": "s2", "text": "Mid",   "tone": "dark",
         "choices": [{"text": "old", "next": "s1"}]},   # broken
        {"id": "s3", "text": "End",   "tone": "hopeful", "choices": []},
    ]
}

patch = {
    "s1": [{"text": "fixed", "next": "s2"}],
    "s2": [{"text": "fixed", "next": "s3"}],
}

merged = merge_patch(original, patch)

s1 = next(s for s in merged["scenes"] if s["id"] == "s1")
s2 = next(s for s in merged["scenes"] if s["id"] == "s2")
s3 = next(s for s in merged["scenes"] if s["id"] == "s3")

chk("B1 s1 choices patched to forward edge",   s1["choices"][0]["next"] == "s2")
chk("B2 s2 choices patched to forward edge",   s2["choices"][0]["next"] == "s3")
chk("B3 s3 (unlisted) unchanged",              s3["choices"] == [])
chk("B4 scene text not mutated",               s1["text"] == "Start")
chk("B5 scene tone not mutated",               s1["tone"] == "tense")
chk("B6 original dict not mutated",
    original["scenes"][0]["choices"][0]["next"] == "s1")
chk("B7 empty patch returns original",         merge_patch(original, {}) is original)
chk("B8 unknown scene_id in patch skipped gracefully",
    merge_patch(original, {"GHOST": []}) is not None)

In [ ]:
# Cell 23 — Test C: orchestrator budget-gate + retry-loop state machine
#
# Simulates the routing logic WITHOUT calling any LLM.
# Two scenarios:
#   C-scenario-1: budget is ample -> retries fire normally -> exhausted at max
#   C-scenario-2: budget is tight -> _has_budget_for_retry returns False -> immediate fail

import os

print("=" * 60)
print("TEST C -- Orchestrator budget-gate + retry-loop simulation")
print("=" * 60)

# Inline the routing helpers (mirrors agents/orchestrator.py exactly)
_REPAIR_RESERVE   = 1_200
_FULL_GEN_RESERVE = 9_500

def _has_budget_for_retry(state):
    ceiling   = int(os.environ.get("TOKEN_BUDGET_LIMIT", 10_000))
    remaining = ceiling - state.get("token_spend", 0)
    will_repair = bool(state.get("broken_nodes")) and state.get("last_story") is not None
    reserve = _REPAIR_RESERVE if will_repair else _FULL_GEN_RESERVE
    return remaining >= reserve

def _route(state):
    if state.get("approved"):
        return "compliance"
    can_retry = state.get("retry_count", 0) < state.get("max_retries", 3) \
                and _has_budget_for_retry(state)
    return "retry" if can_retry else "fail"

def _increment(state):
    return {**state, "retry_count": state.get("retry_count", 0) + 1,
            "story": None, "approved": False}

# -- Scenario 1: ample budget, retries exhaust at max_retries=3 --
os.environ["TOKEN_BUDGET_LIMIT"] = "20000"

state = {
    "approved": False, "retry_count": 0, "max_retries": 3,
    "token_spend": 9000, "broken_nodes": None, "last_story": None,
}
routes = []
for _ in range(6):
    r = _route(state)
    routes.append(r)
    if r == "retry":
        state = _increment(state)
        state["token_spend"] += 600   # simulate repair spend
    elif r == "fail":
        routes.append("compliance")
        break
    elif r == "compliance":
        break

print(f"  Scenario 1 route sequence: {' -> '.join(routes)}")
chk("C1 retry fires when budget ample",        "retry" in routes)
chk("C2 terminates at fail after max_retries",  routes[-2] == "fail")
chk("C3 audit compliance always last",          routes[-1] == "compliance")
chk("C4 retry_count reached max_retries",       state["retry_count"] == 3)

# -- Scenario 2: tight budget -> budget gate fires immediately --
os.environ["TOKEN_BUDGET_LIMIT"] = "10000"

tight_state = {
    "approved": False, "retry_count": 0, "max_retries": 3,
    "token_spend": 9200,   # only 800 remaining, < 9500 full-gen reserve
    "broken_nodes": None, "last_story": None,
}
r = _route(tight_state)
chk("C5 budget gate routes to fail when remaining < reserve", r == "fail", f"got '{r}'")

# -- Scenario 3: repair mode reserve is smaller (1,200) --
repair_state = {
    "approved": False, "retry_count": 1, "max_retries": 3,
    "token_spend": 9200,   # 800 remaining: < 9500 full-gen BUT > 1200 repair
    "broken_nodes": [("s1","s2")],
    "last_story": {"title": "x"},
}
# 800 < 1200, so even repair gate should block
r = _route(repair_state)
chk("C6 repair reserve (1200) also blocks when remaining=800", r == "fail", f"got '{r}'")

# 1500 remaining is enough for repair but not full-gen
repair_state2 = {**repair_state, "token_spend": 8500}  # 1500 remaining
r = _route(repair_state2)
chk("C7 repair gate allows retry when remaining=1500", r == "retry", f"got '{r}'")

# -- Scenario 4: approved=True routes direct to compliance --
approved_state = {**tight_state, "approved": True}
chk("C8 approved=True routes to compliance regardless of budget",
    _route(approved_state) == "compliance")

# Restore env
os.environ["TOKEN_BUDGET_LIMIT"] = "20000"

In [ ]:
# Cell 24 — Test D: Prometheus counters + StoryStore CRUD

print("=" * 60)
print("TEST D -- Telemetry counters & StoryStore")
print("=" * 60)

from core.telemetry import (
    STORIES_GENERATED, LOOP_DETECTIONS, STYLE_VIOLATIONS,
    SANDBOX_REJECTIONS, AGENT_TOKEN_SPEND, BUDGET_HALTS, VALIDATION_DURATION,
)
from core.story_store import StoryStore

def _cv(counter) -> float:
    return counter._value.get()

# Counter increments
before = {
    "gen":   _cv(STORIES_GENERATED),
    "loop":  _cv(LOOP_DETECTIONS),
    "style": _cv(STYLE_VIOLATIONS),
    "rej":   _cv(SANDBOX_REJECTIONS),
    "tok":   _cv(AGENT_TOKEN_SPEND),
    "halt":  _cv(BUDGET_HALTS),
}

STORIES_GENERATED.inc()
LOOP_DETECTIONS.inc(3)
STYLE_VIOLATIONS.inc(2)
SANDBOX_REJECTIONS.inc()
AGENT_TOKEN_SPEND.inc(512)
BUDGET_HALTS.inc()

chk("D1 STORIES_GENERATED +1",       _cv(STORIES_GENERATED) == before["gen"]   + 1)
chk("D2 LOOP_DETECTIONS +3",         _cv(LOOP_DETECTIONS)   == before["loop"]  + 3)
chk("D3 STYLE_VIOLATIONS +2",        _cv(STYLE_VIOLATIONS)  == before["style"] + 2)
chk("D4 SANDBOX_REJECTIONS +1",      _cv(SANDBOX_REJECTIONS)== before["rej"]   + 1)
chk("D5 AGENT_TOKEN_SPEND +512",     _cv(AGENT_TOKEN_SPEND) == before["tok"]   + 512)
chk("D6 BUDGET_HALTS +1",            _cv(BUDGET_HALTS)      == before["halt"]  + 1)

# StoryStore CRUD
store = StoryStore()
store.save("abc", {"title": "Test", "approved": True})
store.save("xyz", {"title": "Demo", "approved": False})

chk("D7 save + get round-trip",      store.get("abc")["title"] == "Test")
chk("D8 all_ids returns both",       set(store.all_ids()) == {"abc", "xyz"}, str(store.all_ids()))
chk("D9 get unknown ID = None",      store.get("MISSING") is None)
chk("D10 delete removes record",
    store.delete("abc") is True and store.get("abc") is None)

# Final scorecard
print()
print("=" * 60)
total = PASS + FAIL
print(f"  RESULT: {PASS}/{total} tests passed", end="  ")
if FAIL == 0:
    print("ALL PASS -- pipeline is structurally sound.")
else:
    print(f"{FAIL} FAILED -- review output above.")
print("=" * 60)